# Pharmacy Analysis

This notebook reads in the input datasets used to run the routing tool for projects relevant to pharmacies, and produces outputs that can be used for further analysis.

* Load LSOA PWC and population data, GP practice and branch data, pharamcy data, and shapefiles for LSOAs, ICBs and parliamentary constituencies.
* Load all outputs from the routing tool.
* Produce a dataset based on the routing from pharmacies to pharmacies by foot. Remove routes of pharmacies to themselves, and find all pharmacies where the closest other pharmacy was more than 0.8 miles away. Identify which ICB each pharmacy is in.
* For driving and walking respectively, collect one row per LSOA, showing how many pharmacies are within 5, 10, 15, etc. minutes of each LSOA's PWC, as well as details about the closest pharmacy (time-wise).
* For driving and walking respectively, produce a dataset of "round trips", i.e. how long it takes to go from an LSOA PWC to a GP to a pharmacy back to the starting LSOA PWC. Identify the 100 quickest routes in this way - treating dispensing GPs as both a GP and a pharmacy. Produce similar summary statistics about these round trips, with one row per LSOA.
* For driving and walking respectively, produce population-weighted averages of ICB and parliamentary constituency summary statistics - using lookups of LSOAs to ICBs and LSOAs to constituencies.
* Plot basic maps of some summary statistics, to act as demonstrations.
* Save datasets to the `output` folder for exporting to Sharepoint.

In [0]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import colors

from src.datavis import find_which_polygon_a_point_is_in, plot_map
from src.pharmacy_analysis.round_trips import (
    get_top_n_round_trips,
    join_round_trips_to_inputs,
    summarise_round_trip_stats,
)
from src.read_write_data import load_config, read_in_route, write_dataset
from src.stats import (
    aggregate_stats_by_geography,
    aggregate_summary_upto_larger_geography,
    bin_values,
    summarise_stats
)

config: dict = load_config("../../config.yml")

## Read in data

In [0]:
lsoa_pwcs: pd.DataFrame = (
    pd.read_csv(config["lsoa_pwc_clean_data_file_path"])
    .merge(
        pd.read_csv(config["lsoa_rural_urban_classification"]),
        how="left",
        left_on="unique_code",
        right_on="LSOA21CD",
        validate="one_to_one",
    )
    .merge(
        pd.read_csv(config["lsoa_population_data"]),
        how="left",
        left_on="unique_code",
        right_on="LSOA 2021 Code"
    )
    .rename(columns={"Total": "population"})
    .assign(population=lambda df: df.population.str.replace(',', '').astype(int))
)[
    ["unique_code", "lat", "lon", "LSOA21NM", "RUC21CD", "RUC21NM", "Urban_rural_flag", "population"]
]
lsoa_pwcs.head(1)

In [0]:
gps: pd.DataFrame = (
    pd.read_csv(config["gps_clean"])
    .merge(
        pd.read_csv(config["dispensing_gps_list_file_path"])[
            ["PRACTICE_CODE", "DISPENSING _GPS"]
        ]
        .rename(
            columns={
                "PRACTICE_CODE": "unique_code", "DISPENSING _GPS": "dispensing_gps"
            }
        ),
        how="left",
        on="unique_code",
    )
    .assign(is_dispensing_gp = lambda df: ~df.dispensing_gps.isna())
    .drop(columns="dispensing_gps")
)
gps.head(1)

In [0]:
pharmacies: pd.DataFrame = pd.read_csv(config["pharmacies_clean"])
pharmacies.head(1)

In [0]:
lsoa_to_icb_df: pd.DataFrame = pd.read_csv(config["lsoa_to_icb"])
lsoa_to_icb_df.head(1)

In [0]:
icb_boundaries_en: gpd.GeoDataFrame = gpd.read_file(
    "../../input/map/ICB_APR_2023_EN_BGC.shp"
)
icb_boundaries_en.head(1)

In [0]:
lsoa_to_constituency_df: pd.DataFrame = (
    gpd.read_file(
        "../../input/map/LSOA_(2021)_to_Westminster_Parliamentary_Constituencies_(July_2024)_Best_Fit_Lookup_in_EW.shp"
    )[["LSOA21CD", "PCON25CD", "PCON25NM"]]
    .rename(columns={"PCON25CD": "PCON24CD", "PCON25NM": "PCON24NM"})
    # drop Welsh LSOAs
    .query("LSOA21CD.str[0] == 'E'")
    .reset_index(drop=True)
)
lsoa_to_constituency_df.head(1)

In [0]:
constituency_boundaries_en: gpd.GeoDataFrame = gpd.read_file(
    "../../input/map/PCON_JULY_2024_UK_BFE.shp"
).query("PCON24CD.str[0] == 'E'")
constituency_boundaries_en.head(1)

In [0]:
lsoa_boundaries_en: gpd.GeoDataFrame = gpd.read_file(
    "../../input/map/LSOA_2021_EW_BSC_V4.shp"
).query("LSOA21CD.str[0] == 'E'")
lsoa_boundaries_en.head(1)

In [0]:
df = pd.concat(
    [
        read_in_route(config["database_path"], "lsoa_pwcs", "gps", "foot"),
        read_in_route(config["database_path"], "lsoa_pwcs", "pharmacies", "foot"),
        read_in_route(config["database_path"], "gps", "pharmacies", "foot"),
        read_in_route(config["database_path"], "pharmacies", "pharmacies", "foot"),
        read_in_route(config["database_path"], "lsoa_pwcs", "gps", "car"),
        read_in_route(config["database_path"], "lsoa_pwcs", "pharmacies", "car"),
        read_in_route(config["database_path"], "gps", "pharmacies", "car"),
    ]
).reset_index(drop=True)
df

## Pharmacy to Pharmacy Analysis

In [0]:
pharmacies_top_3_closest: pd.DataFrame = (
    find_which_polygon_a_point_is_in(
        find_which_polygon_a_point_is_in(
            df
            .query("transportProfile == 'foot'")
            .query("sourceDatasetL == 'pharmacies' == sourceDatasetR")
            .query("uniqueCodeL != uniqueCodeR")
            .sort_values("distance")
            .groupby("uniqueCodeL")
            .head(3)
            .merge(
                pharmacies, how="left", left_on="uniqueCodeL", right_on="unique_code"
            )
            .sort_values(["uniqueCodeL", "distance"]),
            icb_boundaries_en[["ICB23CD", "ICB23NM", "geometry"]],
        )
        .drop(columns=["geometry", "lat", "lon"])
        .rename(columns={"ICB23CD": "ICB23CD_l", "ICB23NM": "ICB23NM_l"})
        .merge(
            pharmacies[["unique_code", "postcode", "trading_name", "organisation_name", "contract_type", "lat", "lon"]],
            how="left",
            left_on="uniqueCodeR",
            right_on="unique_code",
            suffixes=["_l", "_r"],
        )
        .drop(columns=["unique_code_l", "unique_code_r"]),
        icb_boundaries_en[["ICB23CD", "ICB23NM", "geometry"]],
    )
    .drop(columns=["geometry", "lat", "lon"])
    .rename(columns={"ICB23CD": "ICB23CD_r", "ICB23NM": "ICB23NM_r"})
)
pharmacies_top_3_closest.head(1)

## Cars

In [0]:
car_trips: pd.DataFrame = df.query("transportProfile == 'car'")
lsoa_pharmacy_car = car_trips.query(
    "sourceDatasetL == 'lsoa_pwcs' and sourceDatasetR == 'pharmacies'"
)
lsoa_pharmacy_car_stats: pd.DataFrame = summarise_stats(
    lsoa_pharmacy_car, lsoa_pwcs, pharmacies, [5, 10, 15, 20, 30]
).rename(
    columns={
        "lat_lon_location": "lat_lon_pharmacy",
        "trading_name": "pharmacy_name",
        "organisation_name": "pharmacy_org_name"}
)
lsoa_pharmacy_car_stats

In [0]:
aggregate_summary_upto_larger_geography(
    lsoa_pharmacy_car_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_30_minutes",
    "ICB23CD",
)

In [0]:
plot_map(
    lsoa_pharmacy_car_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "number_of_locations_within_15_minutes",
    "LSOA21CD",
    higher_is_worse=False,
    #norm=colors.TwoSlopeNorm(vcenter=30, vmin=0, vmax=50),
)
plot_map(
    lsoa_pharmacy_car_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "min_time",
    "LSOA21CD",
    higher_is_worse=True,
    #norm=colors.TwoSlopeNorm(vcenter=10, vmin=0, vmax=30),
)

In [0]:
lsoa_pwcs.query("unique_code == 'E01019050'")

In [0]:
plot_map(
    lsoa_pharmacy_car_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_15_minutes",
    "ICB23CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_pharmacy_car_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "min_time",
    "ICB23CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    lsoa_pharmacy_car_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "number_of_locations_within_15_minutes",
    "PCON24CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_pharmacy_car_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "min_time",
    "PCON24CD",
    higher_is_worse=True,
    norm=colors.TwoSlopeNorm(vcenter=5, vmin=1, vmax=9)
)

In [0]:
# round_trips_car: pd.DataFrame = get_top_n_round_trips(
#     lsoas_to_gps=car_trips.query("sourceDatasetL == 'lsoa_pwcs' and sourceDatasetR == 'gps'"),
#     lsoas_to_pharmacies=car_trips.query("sourceDatasetL == 'lsoa_pwcs' and sourceDatasetR == 'pharmacies'"),
#     gps_to_pharmacies=car_trips.query("sourceDatasetL == 'gps' and sourceDatasetR == 'pharmacies'"),
#     gps=gps,
# )
# write_dataset(round_trips_car, config["top_100_round_trips_car"], specify_date=False)
round_trips_car = pd.read_csv(config["top_100_round_trips_car"])
round_trips_car

In [0]:
round_trips_car_with_input_data: pd.DataFrame = summarise_round_trip_stats(
    round_trips_car,
    lsoa_pwcs,
    gps,
    pharmacies,
    [5, 10, 15, 20, 30, 40, 50, 60, 75, 90],
    "total_time",
)
round_trips_car_with_input_data.head(1)

In [0]:
plot_map(
    round_trips_car_with_input_data,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "number_of_locations_within_15_minutes",
    "LSOA21CD",
    higher_is_worse=False,
)
plot_map(
    round_trips_car_with_input_data,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "min_time",
    "LSOA21CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    round_trips_car_with_input_data,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_15_minutes",
    "ICB23CD",
    higher_is_worse=False,
)
plot_map(
    round_trips_car_with_input_data,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "min_time",
    "ICB23CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    round_trips_car_with_input_data,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "number_of_locations_within_15_minutes",
    "PCON24CD",
    higher_is_worse=False,
)
plot_map(
    round_trips_car_with_input_data,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "min_time",
    "PCON24CD",
    higher_is_worse=True,
)

In [0]:
sns.histplot(
    bin_values(
        round_trips_car_with_input_data,
        "min_time",
        [0, 5, 10, 15, 20, 25, 30, 40, 50, 60]
    )
    .query("Urban_rural_flag == 'Rural'")[
        "min_time_binned"
    ]
)

In [0]:
(
    bin_values(lsoa_pharmacy_car_stats, "min_time", [0, 5, 10, 15, 20, 25, 30])
    .groupby(["min_time_binned"])
    .agg({"population": sum})
    .reset_index()
    .query("population > 0")
)

In [0]:
(
    bin_values(lsoa_pharmacy_car, "time", [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 150])
    .groupby(["uniqueCodeL", "time_binned"])
    .agg({"uniqueCodeR": len})
    .reset_index()
    .query("~uniqueCodeR.isnull()")
)

## Walking

In [0]:
foot_trips: pd.DataFrame = df.query("transportProfile == 'foot'")
lsoa_pharmacy_foot = foot_trips.query(
    "sourceDatasetL == 'lsoa_pwcs' and sourceDatasetR == 'pharmacies'"
)
lsoa_pharmacy_foot_stats: pd.DataFrame = summarise_stats(
    lsoa_pharmacy_foot, lsoa_pwcs, pharmacies, [5, 10, 15, 20, 30, 40, 50, 60]
).rename(
    columns={
        "lat_lon_location": "lat_lon_pharmacy",
        "trading_name": "pharmacy_name",
        "organisation_name": "pharmacy_org_name"}
)
lsoa_pharmacy_foot_stats

In [0]:
aggregate_summary_upto_larger_geography(
    lsoa_pharmacy_foot_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_30_minutes",
    "ICB23CD",
)

In [0]:
plot_map(
    lsoa_pharmacy_foot_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "number_of_locations_within_30_minutes",
    "LSOA21CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_pharmacy_foot_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "min_time",
    "LSOA21CD",
    higher_is_worse=True,
    norm=colors.TwoSlopeNorm(vcenter=10, vmin=0, vmax=30),
)

In [0]:
plot_map(
    lsoa_pharmacy_foot_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_30_minutes",
    "ICB23CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_pharmacy_foot_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "min_time",
    "ICB23CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    lsoa_pharmacy_foot_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "number_of_locations_within_30_minutes",
    "PCON24CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_pharmacy_foot_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "min_time",
    "PCON24CD",
    higher_is_worse=True,
    # norm=colors.TwoSlopeNorm(vcenter=5, vmin=1, vmax=9)
)

In [0]:
# round_trips_foot: pd.DataFrame = get_top_n_round_trips(
#     lsoas_to_gps=foot_trips.query("sourceDatasetL == 'lsoa_pwcs' and sourceDatasetR == 'gps'"),
#     lsoas_to_pharmacies=foot_trips.query("sourceDatasetL == 'lsoa_pwcs' and sourceDatasetR == 'pharmacies'"),
#     gps_to_pharmacies=foot_trips.query("sourceDatasetL == 'gps' and sourceDatasetR == 'pharmacies'"),
#     gps=gps,
# )
# write_dataset(round_trips_foot, config["top_100_round_trips_foot"], specify_date=False)
round_trips_foot = pd.read_csv(config["top_100_round_trips_foot"])
round_trips_foot

In [0]:
round_trips_foot_with_input_data: pd.DataFrame = summarise_round_trip_stats(
    round_trips_foot,
    lsoa_pwcs,
    gps,
    pharmacies,
    [5, 10, 15, 20, 30, 40, 50, 60, 75, 90],
    "total_time",
)
round_trips_foot_with_input_data.head(1)

In [0]:
plot_map(
    round_trips_foot_with_input_data,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "number_of_locations_within_30_minutes",
    "LSOA21CD",
    higher_is_worse=False,
)
plot_map(
    round_trips_foot_with_input_data,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "min_time",
    "LSOA21CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    round_trips_foot_with_input_data,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_30_minutes",
    "ICB23CD",
    higher_is_worse=False,
)
plot_map(
    round_trips_foot_with_input_data,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "min_time",
    "ICB23CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    round_trips_foot_with_input_data,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "number_of_locations_within_30_minutes",
    "PCON24CD",
    higher_is_worse=False,
)
plot_map(
    round_trips_foot_with_input_data,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "min_time",
    "PCON24CD",
    higher_is_worse=True,
)

In [0]:
_, ax = plt.subplots()
sns.histplot(
    bin_values(
        round_trips_foot_with_input_data,
        "min_time",
        [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 150, 180, 240, 300, 360]
    )
    .query("Urban_rural_flag == 'Urban'")[
        "min_time_binned"
    ]
)
ax.tick_params(axis='x', labelrotation=45)
plt.show()

In [0]:
(
    bin_values(
        lsoa_pharmacy_foot_stats,
        "min_time",
        [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 150, 180, 240, 300, 360, 420])
    .groupby(["min_time_binned"])
    .agg({"population": sum})
    .reset_index()
    .query("population > 0")
)

In [0]:
(
    bin_values(lsoa_pharmacy_foot, "time", [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 150, 180, 240, 300, 360, 420, 480, 540, 600, 2400])
    .groupby(["uniqueCodeL", "time_binned"])
    .agg({"uniqueCodeR": len})
    .reset_index()
    .query("~uniqueCodeR.isnull()")
)

## Output data

In [0]:
lsoa_pharmacy_car_stats_output = (
    lsoa_pharmacy_car_stats
    .merge(
        lsoa_to_icb_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )
    .rename(columns={"LSOA21NM_x": "LSOA21NM"})
    .merge(
        lsoa_to_constituency_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )[[
        "lsoa_id",
        "LSOA21NM",
        "Urban_rural_flag",
        "population",
        "lat_lon_lsoa",
        "SICBL23CD",
        "SICBL23CDH",
        "SICBL23NM",
        "ICB23CD",
        "ICB23CDH",
        "ICB23NM",
        "LAD23CD",
        "LAD23NM",
        "PCON24CD",
        "PCON24NM",
        "min_time",
        "distance",
        "number_of_locations_within_5_minutes",
        "number_of_locations_within_10_minutes",
        "number_of_locations_within_15_minutes",
        "number_of_locations_within_20_minutes",
        "number_of_locations_within_30_minutes",
        "min_time_location_id",
        "pharmacy_name",
        "pharmacy_org_name",
        "contract_type",
        "lat_lon_pharmacy",
    ]]
)
lsoa_pharmacy_car_stats_output.head(1)

In [0]:
lsoa_pharmacy_foot_stats_output = (
    lsoa_pharmacy_foot_stats
    .merge(
        lsoa_to_icb_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )
    .rename(columns={"LSOA21NM_x": "LSOA21NM"})
    .merge(
        lsoa_to_constituency_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )[[
        "lsoa_id",
        "LSOA21NM",
        "Urban_rural_flag",
        "population",
        "lat_lon_lsoa",
        "SICBL23CD",
        "SICBL23CDH",
        "SICBL23NM",
        "ICB23CD",
        "ICB23CDH",
        "ICB23NM",
        "LAD23CD",
        "LAD23NM",
        "PCON24CD",
        "PCON24NM",
        "min_time",
        "distance",
        "number_of_locations_within_5_minutes",
        "number_of_locations_within_10_minutes",
        "number_of_locations_within_15_minutes",
        "number_of_locations_within_20_minutes",
        "number_of_locations_within_30_minutes",
        "min_time_location_id",
        "pharmacy_name",
        "pharmacy_org_name",
        "contract_type",
        "lat_lon_pharmacy",
    ]]
)
lsoa_pharmacy_foot_stats_output.head(1)

In [0]:
round_trips_car_summary_output = (
    round_trips_car_with_input_data
    .merge(
        lsoa_to_icb_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )
    .rename(columns={"LSOA21NM_x": "LSOA21NM"})
    .merge(
        lsoa_to_constituency_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )[[
        "lsoa_id",
        "LSOA21NM",
        "Urban_rural_flag",
        "population",
        "lat_lon_lsoa",
        "SICBL23CD",
        "SICBL23CDH",
        "SICBL23NM",
        "ICB23CD",
        "ICB23CDH",
        "ICB23NM",
        "LAD23CD",
        "LAD23NM",
        "PCON24CD",
        "PCON24NM",
        "min_time",
        "total_distance",
        "number_of_locations_within_5_minutes",
        "number_of_locations_within_10_minutes",
        "number_of_locations_within_15_minutes",
        "number_of_locations_within_20_minutes",
        "number_of_locations_within_30_minutes",
        "number_of_locations_within_40_minutes",
        "number_of_locations_within_50_minutes",
        "number_of_locations_within_60_minutes",
        "number_of_locations_within_75_minutes",
        "number_of_locations_within_90_minutes",
        "gp_id",
        "gp_name",
        "is_dispensing_gp",
        "number_of_dispensing_gps",
        "lat_lon_gp",
        "pharmacy_id",
        "pharmacy_name",
        "pharmacy_org_name",
        "contract_type",
        "lat_lon_pharmacy",
    ]]
    .rename(columns={
        k: k.replace("locations", "routes")
        for k in round_trips_car_with_input_data.columns
        if "locations" in k
    })
    .astype({
        "lat_lon_lsoa": str,
        "lat_lon_gp": str,
        "lat_lon_pharmacy": str,
    })
    .replace("(nan, nan)", np.NaN)
)
round_trips_car_summary_output.head(1)

In [0]:
round_trips_foot_summary_output = (
    round_trips_foot_with_input_data
    .merge(
        lsoa_to_icb_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )
    .rename(columns={"LSOA21NM_x": "LSOA21NM"})
    .merge(
        lsoa_to_constituency_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )[[
        "lsoa_id",
        "LSOA21NM",
        "Urban_rural_flag",
        "population",
        "lat_lon_lsoa",
        "SICBL23CD",
        "SICBL23CDH",
        "SICBL23NM",
        "ICB23CD",
        "ICB23CDH",
        "ICB23NM",
        "LAD23CD",
        "LAD23NM",
        "PCON24CD",
        "PCON24NM",
        "min_time",
        "total_distance",
        "number_of_locations_within_5_minutes",
        "number_of_locations_within_10_minutes",
        "number_of_locations_within_15_minutes",
        "number_of_locations_within_20_minutes",
        "number_of_locations_within_30_minutes",
        "number_of_locations_within_40_minutes",
        "number_of_locations_within_50_minutes",
        "number_of_locations_within_60_minutes",
        "number_of_locations_within_75_minutes",
        "number_of_locations_within_90_minutes",
        "gp_id",
        "gp_name",
        "is_dispensing_gp",
        "number_of_dispensing_gps",
        "lat_lon_gp",
        "pharmacy_id",
        "pharmacy_name",
        "pharmacy_org_name",
        "contract_type",
        "lat_lon_pharmacy",
    ]]
    .rename(columns={
        k: k.replace("locations", "routes")
        for k in round_trips_car_with_input_data.columns
        if "locations" in k
    })
    .astype({
        "lat_lon_lsoa": str,
        "lat_lon_gp": str,
        "lat_lon_pharmacy": str,
    })
    .replace("(nan, nan)", np.NaN)
)
round_trips_foot_summary_output.head(1)

In [0]:
lsoa_pharmacy_car_stats_ICB_output = (
    lsoa_to_icb_df[["ICB23CD", "ICB23NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_pharmacy_car_stats_output,
            "ICB23CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="ICB23CD",
    )
)
lsoa_pharmacy_car_stats_ICB_output.head(1)

In [0]:
lsoa_pharmacy_car_stats_PCON_output = (
    lsoa_to_constituency_df[["PCON24CD", "PCON24NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_pharmacy_car_stats_output,
            "PCON24CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="PCON24CD",
    )
)
lsoa_pharmacy_car_stats_PCON_output.head(1)

In [0]:
lsoa_pharmacy_foot_stats_ICB_output = (
    lsoa_to_icb_df[["ICB23CD", "ICB23NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_pharmacy_foot_stats_output,
            "ICB23CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="ICB23CD",
    )
)
lsoa_pharmacy_foot_stats_ICB_output.head(1)

In [0]:
lsoa_pharmacy_foot_stats_PCON_output = (
    lsoa_to_constituency_df[["PCON24CD", "PCON24NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_pharmacy_foot_stats_output,
            "PCON24CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="PCON24CD",
    )
)
lsoa_pharmacy_foot_stats_PCON_output.head(1)

In [0]:
round_trips_car_summary_ICB_output = (
    lsoa_to_icb_df[["ICB23CD", "ICB23NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            round_trips_car_summary_output,
            "ICB23CD",
            [
                "min_time",
                # "distance",
                "number_of_routes_within_5_minutes",
                "number_of_routes_within_10_minutes",
                "number_of_routes_within_15_minutes",
                "number_of_routes_within_20_minutes",
                "number_of_routes_within_30_minutes",
                "number_of_routes_within_40_minutes",
                "number_of_routes_within_50_minutes",
                "number_of_routes_within_60_minutes",
                "number_of_routes_within_75_minutes",
                "number_of_routes_within_90_minutes",
            ],
        ),
        how="left",
        on="ICB23CD",
    )
)
round_trips_car_summary_ICB_output.head(1)

In [0]:
round_trips_car_summary_PCON_output = (
    lsoa_to_constituency_df[["PCON24CD", "PCON24NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            round_trips_car_summary_output,
            "PCON24CD",
            [
                "min_time",
                # "distance",
                "number_of_routes_within_5_minutes",
                "number_of_routes_within_10_minutes",
                "number_of_routes_within_15_minutes",
                "number_of_routes_within_20_minutes",
                "number_of_routes_within_30_minutes",
                "number_of_routes_within_40_minutes",
                "number_of_routes_within_50_minutes",
                "number_of_routes_within_60_minutes",
                "number_of_routes_within_75_minutes",
                "number_of_routes_within_90_minutes",
            ],
        ),
        how="left",
        on="PCON24CD",
    )
)
round_trips_car_summary_PCON_output.head(1)

In [0]:
round_trips_foot_summary_ICB_output = (
    lsoa_to_icb_df[["ICB23CD", "ICB23NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            round_trips_foot_summary_output,
            "ICB23CD",
            [
                "min_time",
                # "distance",
                "number_of_routes_within_5_minutes",
                "number_of_routes_within_10_minutes",
                "number_of_routes_within_15_minutes",
                "number_of_routes_within_20_minutes",
                "number_of_routes_within_30_minutes",
                "number_of_routes_within_40_minutes",
                "number_of_routes_within_50_minutes",
                "number_of_routes_within_60_minutes",
                "number_of_routes_within_75_minutes",
                "number_of_routes_within_90_minutes",
            ],
        ),
        how="left",
        on="ICB23CD",
    )
)
round_trips_foot_summary_ICB_output.head(1)

In [0]:
round_trips_foot_summary_PCON_output = (
    lsoa_to_constituency_df[["PCON24CD", "PCON24NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            round_trips_foot_summary_output,
            "PCON24CD",
            [
                "min_time",
                # "distance",
                "number_of_routes_within_5_minutes",
                "number_of_routes_within_10_minutes",
                "number_of_routes_within_15_minutes",
                "number_of_routes_within_20_minutes",
                "number_of_routes_within_30_minutes",
                "number_of_routes_within_40_minutes",
                "number_of_routes_within_50_minutes",
                "number_of_routes_within_60_minutes",
                "number_of_routes_within_75_minutes",
                "number_of_routes_within_90_minutes",
            ],
        ),
        how="left",
        on="PCON24CD",
    )
)
round_trips_foot_summary_PCON_output.head(1)

In [0]:
write_dataset(
    lsoa_pharmacy_car
    .merge(
        lsoa_to_icb_df[["LSOA21CD", "ICB23CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD")
    .merge(
        lsoa_to_constituency_df[["LSOA21CD", "PCON24CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD"),
    "../../output/full_output_LSOAs_to_pharmacies_car_with_ICB_PCON.csv"
)

In [0]:
write_dataset(
    lsoa_pharmacy_foot
    .merge(
        lsoa_to_icb_df[["LSOA21CD", "ICB23CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD")
    .merge(
        lsoa_to_constituency_df[["LSOA21CD", "PCON24CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD"),
    "../../output/full_output_LSOAs_to_pharmacies_foot_with_ICB_PCON.csv"
)

In [0]:
write_dataset(
    lsoa_pharmacy_car_stats_output,
    "../../output/summary_data_LSOAs_to_pharmacies_car.csv"
)

In [0]:
write_dataset(
    lsoa_pharmacy_foot_stats_output,
    "../../output/summary_data_LSOAs_to_pharmacies_foot.csv"
)

In [0]:
write_dataset(
    round_trips_car_summary_output,
    "../../output/summary_data_LSOAs_to_GPs_to_pharmacies_car.csv",
)

In [0]:
write_dataset(
    round_trips_foot_summary_output,
    "output/summary_data_LSOAs_to_GPs_to_pharmacies_foot.csv",
)

In [0]:
write_dataset(
    lsoa_pharmacy_car_stats_ICB_output,
    "../../output/summary_data_ICBs_to_pharmacies_car.csv"
)

In [0]:
write_dataset(
    lsoa_pharmacy_car_stats_PCON_output,
    "../../output/summary_data_PCONs_to_pharmacies_car.csv"
)

In [0]:
write_dataset(
    lsoa_pharmacy_foot_stats_ICB_output,
    "../../output/summary_data_ICBs_to_pharmacies_foot.csv"
)

In [0]:
write_dataset(
    lsoa_pharmacy_foot_stats_PCON_output,
    "../../output/summary_data_PCONs_to_pharmacies_foot.csv"
)

In [0]:
write_dataset(
    round_trips_car,
    "../../output/full_output_LSOAs_to_GPs_to_pharmacies_car.csv"
)

In [0]:
write_dataset(
    round_trips_foot,
    "../../output/full_output_LSOAs_to_GPs_to_pharmacies_foot.csv"
)

In [0]:
write_dataset(
    round_trips_car_summary_ICB_output,
    "../../output/summary_data_ICBs_to_GPs_to_pharmacies_car.csv",
)

In [0]:
write_dataset(
    round_trips_car_summary_PCON_output,
    "../../output/summary_data_PCONs_to_GPs_to_pharmacies_car.csv",
)

In [0]:
write_dataset(
    round_trips_foot_summary_ICB_output,
    "../../output/summary_data_ICBs_to_GPs_to_pharmacies_foot.csv",
)

In [0]:
write_dataset(
    round_trips_foot_summary_PCON_output,
    "../../output/summary_data_PCONs_to_GPs_to_pharmacies_foot.csv",
)

In [0]:
write_dataset(
    pharmacies_top_3_closest,
    "../../output/top_3_closest_pharmacies_to_each_pharmacy_foot.csv",
)